# 분류기 만들기

타이타닉 데이터의 생존여부 분류
- 규칙 : 성별(sex) = 1 사망으로 분류

In [20]:
from sklearn.base import BaseEstimator
import numpy as np

class MyDummyClassifier(BaseEstimator):
    def fit(self, X, y):
        pass
    
    def predict(self, X):
        pred = np.zeros((X.shape[0],1))
        for i in range(X.shape[0]):
            if X['Sex'].iloc[i] == 1:
                pred[i] = 0
            else:
                pred[i] = 1
        return pred

# 타이타닉 데이터 가져오기

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
titnic_df = pd.read_csv("./data/titanic.csv")

In [6]:
y_titanic_df = titnic_df['Survived']
X_titanic_df = titnic_df.drop('Survived', axis=1)

In [8]:
from sklearn.preprocessing import LabelEncoder

# Null 처리 함수
def fillna(df):
    df['Age'].fillna(df['Age'].mean(), inplace=True)
    df['Cabin'].fillna('N', inplace=True)
    df['Embarked'].fillna('N', inplace=True)
    df['Fare'].fillna(0, inplace=True)
    return df

# 머신러닝 알고리즘에 불필요한 피처 제거
def drop_features(df):
    df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
    return df

# 레이블 인코딩 수행 함수
def format_features(df):
    df['Cabin'] = df['Cabin'].str[:1]
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
    df = fillna(df) 
    df = drop_features(df)
    df = format_features(df)
    return df

In [11]:
X_titanic_df = transform_features(X_titanic_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_19936\2811512165.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].mean(), inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_19936\2811512165.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

KeyError: "['PassengerId', 'Name', 'Ticket'] not found in axis"

In [12]:
#데이터셋 분할
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df, y_titanic_df, test_size=0.2, random_state=0)

In [21]:
myclf = MyDummyClassifier()
myclf.fit(X_train, y_train)
my_pred = myclf.predict(X_test)
accuracy_score(y_test, my_pred)

0.7877094972067039

In [22]:
X_titanic_df.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,3,1,22.0,1,0,7.2500,7,3
1,1,0,38.0,1,0,71.2833,2,0
2,3,0,26.0,0,0,7.9250,7,3
3,1,0,35.0,1,0,53.1000,2,3
4,3,1,35.0,0,0,8.0500,7,3


In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, my_pred)


array([[92, 18],
       [20, 49]])

In [24]:
from sklearn.metrics import precision_score, recall_score
precision_score(y_test, my_pred), recall_score(y_test, my_pred)

(np.float64(0.7313432835820896), np.float64(0.7101449275362319))

# 로지스틱회귀, 랜덤포레스트, KNN의 정밀도, 재현율 비교하기

In [25]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [26]:
def get_clf_eval(y_test, pred):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)

    print(confusion)
    print(''*20)
    print(accuracy, precision, recall)

In [ ]:
# 로지스틱회귀로 분류모델 생성
lr_clf = LogisticRegression(max_iter=2000)
lr_clf.fit(X_train, y_train)
pred = lr_clf.predict(X_test)

# 정확도, 정밀도, 재현율
get_clf_eval(y_test, pred)

[[92 18]
 [16 53]]

0.8100558659217877 0.7464788732394366 0.7681159420289855


# 정밀도와 재현율의 변화

정밀도와 재현율의 불균형이 심할 때,<br>
혹은 비지니스의 요구사항이 있을 때<br>
임계치를 조정해야 한다.<br>

임계치를 낮추면 정밀도는 낮아지고, 재현율을 올라간다.

In [81]:
pred_proba = lr_clf.predict_proba(X_test)
pos_proba = pred_proba[:,1] #양성클래스일 확률

threshold = 0.465 #임계치
custom_proba = (pos_proba >= threshold).astype(int) #임계치보다 크면 1
get_clf_eval(y_test, custom_proba)

[[92 18]
 [14 55]]

0.8212290502793296 0.7534246575342466 0.7971014492753623


# 평가결과 확인하기

In [ ]:
from sklearn.metrics import f1_score
f1_score(y_test, custom_proba)

np.float64(0.7746478873239436)

In [82]:
from sklearn.metrics import classification_report
print(classification_report(y_test, custom_proba))

              precision    recall  f1-score   support

           0       0.87      0.84      0.85       110
           1       0.75      0.80      0.77        69

    accuracy                           0.82       179
   macro avg       0.81      0.82      0.81       179
weighted avg       0.82      0.82      0.82       179



In [ ]:
pd.Series(lr_clf.coef_[0]).sort_values() # 피처의 중요도는 계수

1   -2.593416
0   -0.901628
3   -0.368137
7   -0.107352
4   -0.059052
6   -0.058762
2   -0.042756
5    0.001286
dtype: float64

In [ ]:
# 랜덤포레스트로 분류모델 생성
rf_clf = RandomForestClassifier()
rf_clf.fit(X_train, y_train)
pred = rf_clf.predict(X_test)

# 정확도, 정밀도, 재현율
get_clf_eval(y_test, pred)

[[98 12]
 [20 49]]

0.8212290502793296 0.8032786885245902 0.7101449275362319


In [ ]:
# KNN으로 분류모델 생성
kn_clf = KNeighborsClassifier(n_neighbors=5)
kn_clf.fit(X_train, y_train)
pred = kn_clf.predict(X_test)

# 정확도, 정밀도, 재현율
get_clf_eval(y_test, pred)

[[94 16]
 [31 38]]

0.7374301675977654 0.7037037037037037 0.5507246376811594
